In [9]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [37]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [38]:
DATA_DIR = Path("c:\\Users\\Niraj Mhatre\\projects\\motor_insurance_data\\Fast_Furious_Insured")

TRAIN_IMG_DIR = DATA_DIR / "trainImages"

TRAIN_METADATA = (
    DATA_DIR
    / "processed"
    / "train_metadata_clean.csv"
)

df = pd.read_csv(TRAIN_METADATA)

print(df.shape)
df.head()

(1399, 8)


,Image_path,Insurance_company,Cost_of_vehicle,Min_coverage,Expiry_date,Max_coverage,Condition,Amount
0,img_4513976.jpg,BQ,41500.0,1037.5,2026-12-03,36142.68,0,0.0
1,img_7764995.jpg,BQ,50700.0,1267.5,2025-07-10,12753.00,1,6194.0
2,img_451308.jpg,A,49500.0,1237.5,2022-08-11,43102.68,0,0.0
3,img_7768372.jpg,A,33500.0,837.5,2022-08-02,8453.00,1,7699.0
4,img_7765274.jpg,AC,27600.0,690.0,2026-05-01,6978.00,1,8849.0


In [39]:
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Condition"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

Train: (1119, 8)
Validation: (280, 8)


In [40]:
print("Training:")
print(train_df["Condition"].value_counts(normalize=True))

print("\nValidation:")
print(val_df["Condition"].value_counts(normalize=True))

Training:
Condition
1    0.929401
0    0.070599
Name: proportion, dtype: float64

Validation:
Condition
1    0.928571
0    0.071429
Name: proportion, dtype: float64


In [41]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        224,
        scale=(0.75, 1.0),
        ratio=(0.9, 1.1)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=15
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.15,
        hue=0.03
    ),

    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=3)],
        p=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [42]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [43]:
class VehicleDamageDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = self.image_dir / row["Image_path"]

        image = Image.open(image_path).convert("RGB")

        label = torch.tensor(
            row["Condition"],
            dtype=torch.float32
        )

        if self.transform:
            image = self.transform(image)

        return image, label

In [44]:
train_dataset = VehicleDamageDataset(
    train_df,
    TRAIN_IMG_DIR,
    train_transform
)

val_dataset = VehicleDamageDataset(
    val_df,
    TRAIN_IMG_DIR,
    val_transform
)

In [45]:
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [46]:
weights = models.ResNet50_Weights.DEFAULT

model = models.resnet50(weights=weights)

In [47]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    1
)

model = model.to(device)

In [48]:
train_df["Condition"].value_counts()

Condition
1    1040
0      79
Name: count, dtype: int64

In [49]:
num_negative = (train_df["Condition"] == 0).sum()
num_positive = (train_df["Condition"] == 1).sum()

pos_weight = torch.tensor(
    [num_negative / num_positive],
    dtype=torch.float32
).to(device)

print("Positive class weight:", pos_weight.item())

Positive class weight: 0.07596153765916824


In [50]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

In [51]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [52]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [53]:
def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [54]:
def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(outputs)

            all_probs.extend(
                probs.cpu().numpy().ravel()
            )

            all_labels.extend(
                labels.cpu().numpy().ravel()
            )

    loss = running_loss / len(loader.dataset)

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    predictions = (all_probs >= 0.5).astype(int)

    accuracy = accuracy_score(
        all_labels,
        predictions
    )

    precision = precision_score(
        all_labels,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        predictions,
        zero_division=0
    )

    auc = roc_auc_score(
        all_labels,
        all_probs
    )

    return {
        "loss": loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "probs": all_probs,
        "labels": all_labels
    }

In [55]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory:", MODEL_DIR.resolve())

Model directory: C:\Users\Niraj Mhatre\projects\Motor-Insurance-Claim-Severity-Assessment\models


In [56]:
EPOCHS = 15
best_f1 = 0

history = []

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_metrics = evaluate(
        model,
        val_loader,
        criterion
    )

    scheduler.step(val_metrics["f1"])

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"F1: {val_metrics['f1']:.4f} | "
        f"Precision: {val_metrics['precision']:.4f} | "
        f"Recall: {val_metrics['recall']:.4f} | "
        f"AUC: {val_metrics['auc']:.4f}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "f1": val_metrics["f1"],
        "precision": val_metrics["precision"],
        "recall": val_metrics["recall"],
        "auc": val_metrics["auc"]
    })

    if val_metrics["f1"] > best_f1:

        best_f1 = val_metrics["f1"]

        torch.save(
           model.state_dict(),
           MODEL_DIR / "resnet50_best.pth"
        )

        print("  → Best model saved!")

c:\Users\Niraj Mhatre\projects\Motor-Insurance-Claim-Severity-Assessment\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/15 | Train Loss: 0.0727 | Val Loss: 0.4685 | F1: 0.9511 | Precision: 0.9681 | Recall: 0.9346 | AUC: 0.8295
  → Best model saved!


c:\Users\Niraj Mhatre\projects\Motor-Insurance-Claim-Severity-Assessment\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 2/15 | Train Loss: 0.0584 | Val Loss: 0.1023 | F1: 0.8814 | Precision: 0.9811 | Recall: 0.8000 | AUC: 0.8508


c:\Users\Niraj Mhatre\projects\Motor-Insurance-Claim-Severity-Assessment\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Epoch 13/15 | Train Loss: 0.0092 | Val Loss: 0.1279 | F1: 0.9670 | Precision: 0.9765 | Recall: 0.9577 | AUC: 0.8462
  → Best model saved!


c:\Users\Niraj Mhatre\projects\Motor-Insurance-Claim-Severity-Assessment\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 14/15 | Train Loss: 0.0054 | Val Loss: 0.1291 | F1: 0.9731 | Precision: 0.9731 | Recall: 0.9731 | AUC: 0.8379
  → Best model saved!


c:\Users\Niraj Mhatre\projects\Motor-Insurance-Claim-Severity-Assessment\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 15/15 | Train Loss: 0.0070 | Val Loss: 0.1298 | F1: 0.9589 | Precision: 0.9761 | Recall: 0.9423 | AUC: 0.8525


In [5]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Recreate the same ResNet50 architecture
model = models.resnet50(weights=None)

# IMPORTANT: checkpoint has 6 output classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 6)

model = model.to(device)

# Load saved model
checkpoint_path = "resnet50_best.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

# Load state dictionary
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

print("✓ ResNet50 checkpoint loaded successfully.")
print("✓ Number of output classes:", model.fc.out_features)

Device: cpu
✓ ResNet50 checkpoint loaded successfully.
✓ Number of output classes: 6


In [11]:
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

NameError: name 'val_dataset' is not defined

In [10]:
# Inspect the class labels used by the test dataset

print("Dataset type:")
print(type(test_loader.dataset))

print("\nDataset attributes:")

for attr in ["classes", "class_names", "class_to_idx", "idx_to_class"]:
    if hasattr(test_loader.dataset, attr):
        print(f"\n{attr}:")
        print(getattr(test_loader.dataset, attr))

Dataset type:


NameError: name 'test_loader' is not defined